# Faruq-v3 — predicted-ROI multilevel transfer audit

Menguji apakah gain P3+P4+P5 bertahan pada box prediksi D0 dan setelah kapasitas descriptor disamakan menjadi PCA-128. Detector tetap beku, cache tersimpan di shared Drive, dan test tidak diekstrak.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, subprocess, sys, time
from pathlib import Path
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import tarfile
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
    'experiments/faruq-v3-pyramid-separability-v1/pyramid_separability.json',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
CHECKPOINT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
GT_REPORT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-pyramid-separability-v1/pyramid_separability.json')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
OUTPUT = PROJECT_ROOT / 'experiments/faruq-v3-predicted-roi-transfer-v1/predicted_roi_transfer.json'
if not (DATA_ROOT / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive: archive.extractall('/content', filter='data')
assert (DATA_ROOT / 'train/images').is_dir()
assert (DATA_ROOT / 'val/images').is_dir()
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
print('PROJECT   :', PROJECT_ROOT)
print('CHECKPOINT:', CHECKPOINT)
print('GT REPORT :', GT_REPORT)
print('OUTPUT    :', OUTPUT)

In [ ]:
from coffee_detector.analysis.faruq_v3_predicted_roi_transfer import run_faruq_v3_predicted_roi_transfer
result = run_faruq_v3_predicted_roi_transfer(
    CHECKPOINT, DATA_ROOT, GT_REPORT, OUTPUT, device='0', image_size=640, roi_size=3,
    candidate_count=500, iou_threshold=0.50, pca_components=128, ridge=0.01
)
assert result['detector_training_executed'] is False
assert result['probe_fitting_executed'] is True
assert result['pca_fitting_executed'] is True
assert result['validation_images_accessed'] is True
assert result['test_images_accessed'] is False
print('AUDIT SELESAI')

In [ ]:
import pandas as pd
from IPython.display import display
rows = []
for name, values in result['results'].items():
    val = values['validation']
    rows.append({
        'representation': name,
        'dimensions': values['dimensions'],
        'train_macro_f1': values['train']['macro_f1'],
        'val_macro_f1': val['macro_f1'],
        'val_balanced_accuracy': val['balanced_accuracy'],
        'val_bottom3_f1': val['bottom3_f1'],
        'val_worst_f1': val['worst_class_f1'],
        'val_top3_accuracy': val['top3_accuracy'],
        'generalization_gap': values['macro_f1_generalization_gap'],
    })
table = pd.DataFrame(rows)
display(table.style.format({column: '{:.2%}' for column in table.columns if column not in ('representation', 'dimensions')}))
best = 'P3+P4+P5_CM128'
per_class = pd.DataFrame(result['results'][best]['validation']['per_class']).sort_values(['f1', 'class_name'])
print('COVERAGE:', result['coverage'])
print('BOTTOM-10:', best)
display(per_class.head(10).style.format({'f1': '{:.2%}'}))
print('DECISION:', result['decision']['decision'])
print('NEXT:', result['decision']['next_action'])
print('DETAIL:', result['decision'])
print('SUMMARY:', result['summary'])
print('Kirim tabel, coverage, bottom-10, dan keputusan. Jangan training detector.')